In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import numpy as np

results_path = "../results/"
h3_path = os.path.join(results_path, "h3/*.csv")

df_smax_120 = pd.read_csv(os.path.join(results_path, "solution_quality_test_smax_1.2.csv"))
df_smax_125 = pd.read_csv(os.path.join(results_path, "solution_quality_test_smax_1.25.csv"))
df_smax_inf = pd.read_csv(os.path.join(results_path, "solution_quality_test_smax_inf.csv"))

h3_result_files = glob.glob(h3_path)
df_h3 = pd.concat([pd.read_csv(f).assign(Instance=os.path.basename(f).replace('_h3.csv', '')) for f in h3_result_files])

sns.set_theme(style="whitegrid")

In [ ]:
stats = df_smax_125.groupby(['Instance', 'Algorithm'])['Distance'].agg(['min', 'mean', 'std']).reset_index()
stats.columns = ['Instance', 'Algorithm', 'Best', 'Average', 'Std_Dev']

display(stats.sort_values(by=['Instance', 'Best']))

In [ ]:
def plot_instance_boxplots(df, instance_name):
    plt.figure(figsize=(10, 6))
    data = df[df['Instance'] == instance_name].where(df['Algorithm'].isin(['MMAS', 'ASRank', 'BasicAntSystem']))
    sns.boxplot(data=data, x='Algorithm', y='Distance', palette='Set2', hue="Algorithm")
    plt.title(f"Variation of results for instance: {instance_name} (S_max 1.25)")
    plt.show()

unique_instances = df_smax_125['Instance'].unique()
for inst in unique_instances:
    plot_instance_boxplots(df_smax_125, inst)

In [ ]:
def plot_convergence(df_h3, instance_name):
    plt.figure(figsize=(12, 6))
    
    inst_data = df_h3[df_h3['Instance'] == instance_name]

    sns.lineplot(data=inst_data, x='Iteration', y='Best_Cost', hue='Algorithm', errorbar='sd')
    
    plt.axvline(x=40, color='red', linestyle='--', label='20% of iterations')
    
    plt.title(f"Convergence Plot: {instance_name}")
    plt.ylabel("Best Found Cost")
    plt.legend()
    plt.show()

for inst in df_h3['Instance'].unique():
    plot_convergence(df_h3, inst)

In [ ]:
h1_data = df_smax_125.groupby(['Instance', 'Algorithm'])['Distance'].mean().unstack()

h1_data['Threshold_85pct'] = h1_data['Greedy'] * 0.85

for algo in ['BasicAntSystem', 'MMAS', 'ASRank']:
    h1_data[f'{algo}_Satisfies_H1'] = h1_data[algo] <= h1_data['Threshold_85pct']

verification_h1 = h1_data[[algo + '_Satisfies_H1' for algo in ['BasicAntSystem', 'MMAS', 'ASRank']]]
display(verification_h1)

success_rate = verification_h1.mean() * 100
print(f"Percentage of instances satisfying H1:\n{success_rate}")
print(f"Decision: {'CONFIRMED' if (success_rate >= 90).all() else 'REJECTED'}")

In [ ]:
h2_std = df_smax_125[df_smax_125['Algorithm'] != 'Greedy'].groupby(['Instance', 'Algorithm'])['Distance'].std().unstack()

h2_std['Min_STD'] = h2_std.min(axis=1)
h2_std['Is_MMAS_Most_Stable'] = h2_std['MMAS'] == h2_std['Min_STD']

display(h2_std)

total_mmas_wins = h2_std['Is_MMAS_Most_Stable'].mean() * 100

print(f"MMAS was most stable in {total_mmas_wins:.2f}% of all instances.")
print(f"Decision (threshold 80%): {'CONFIRMED' if total_mmas_wins >= 80 else 'REJECTED'}")

In [ ]:
df_40_raw = df_h3[df_h3['Iteration'] == 40].copy()

h3_raw_table = df_40_raw.groupby(['Instance', 'Algorithm'])['Best_Cost'].mean().unstack()

h3_raw_table['Threshold_H3_Original'] = h3_raw_table['BasicAntSystem'] * 0.95

h3_raw_table['Ratio_Raw'] = h3_raw_table['ASRank'] / h3_raw_table['BasicAntSystem']
h3_raw_table['H3_Original_Satisfied'] = h3_raw_table['ASRank'] <= h3_raw_table['Threshold_H3_Original']

display(h3_raw_table[['BasicAntSystem', 'ASRank', 'Ratio_Raw', 'H3_Original_Satisfied']])
h3_raw_success_rate = h3_raw_table['H3_Original_Satisfied'].mean() * 100
print(f"\nH3 (initial version) satisfied for {h3_raw_success_rate:.2f}% of instances.")

In [ ]:
bks_values = {
    "A-n32-k5": 784,
    "A-n33-k6": 742,
    "A-n36-k5": 799,
    "A-n37-k6": 949,
    "A-n39-k5": 822,
    "A-n45-k7": 1146,
    "A-n55-k9": 1073,
    "A-n60-k9": 1354,
    "A-n63-k10": 1314,
    "A-n64-k9": 1401,
    "A-n65-k9": 1174,
    "A-n80-k10": 1763
}

df_40 = df_h3[df_h3['Iteration'] == 40].copy()

h3_table = df_40.groupby(['Instance', 'Algorithm'])['Best_Cost'].mean().unstack()

h3_table['BKS'] = h3_table.index.map(bks_values)
h3_table['Gap_AS'] = h3_table['BasicAntSystem'] - h3_table['BKS']
h3_table['Gap_ASRank'] = h3_table['ASRank'] - h3_table['BKS']

h3_table['Ratio'] = h3_table['Gap_ASRank'] / h3_table['Gap_AS']
h3_table['H3_Satisfied'] = h3_table['Ratio'] <= 0.95

display(h3_table[['Gap_AS', 'Gap_ASRank', 'Ratio', 'H3_Satisfied']])
h3_success_rate = h3_table['H3_Satisfied'].mean() * 100
print(f"\nH3 (updated version) satisfied for {h3_success_rate:.2f}% of instances.")

In [ ]:
mean_125 = df_smax_125.groupby(['Instance', 'Algorithm'])['Distance'].mean().unstack()
mean_120 = df_smax_120.groupby(['Instance', 'Algorithm'])['Distance'].mean().unstack()

h4_data = pd.DataFrame(index=mean_125.index)

h4_data['Greedy_Dist_125'] = mean_125['Greedy']
h4_data['Greedy_Dist_120'] = mean_120['Greedy']
h4_data['Greedy_Delta'] = mean_120['Greedy'] - mean_125['Greedy']
h4_data['Greedy_Change_Pct'] = (h4_data['Greedy_Delta'] / mean_125['Greedy']) * 100

algos = ['BasicAntSystem', 'MMAS', 'ASRank']

for algo in algos:
    q_125 = (mean_125['Greedy'] - mean_125[algo]) / mean_125[algo]
    q_120 = (mean_120['Greedy'] - mean_120[algo]) / mean_120[algo]
    
    h4_data[f'Quality_{algo}_125'] = q_125
    h4_data[f'Quality_{algo}_120'] = q_120
    h4_data[f'Delta_PP_{algo}'] = (q_120 - q_125) * 100
    h4_data[f'{algo}_Satisfies_H4'] = h4_data[f'Delta_PP_{algo}'] >= 5

h4_cols = [f'{algo}_Satisfies_H4' for algo in algos]
h4_data['Instance_Satisfies_H4'] = h4_data[h4_cols].sum(axis=1) >= 2

display_cols = ['Greedy_Delta', 'Greedy_Change_Pct'] + \
               [f'Delta_PP_{algo}' for algo in algos] + \
               ['Instance_Satisfies_H4']
formatted_h4 = h4_data[display_cols].style.format({
    'Greedy_Delta': "{:.2f}",
    'Greedy_Change_Pct': "{:.2f}%",
    'Delta_PP_BasicAntSystem': "{:.2f} p.p.",
    'Delta_PP_MMAS': "{:.2f} p.p.",
    'Delta_PP_ASRank': "{:.2f} p.p."
})
display(formatted_h4)

success_rate = h4_data['Instance_Satisfies_H4'].mean() * 100
print(f"Percentage of instances satisfying H4 condition: {success_rate:.2f}%")
print(f"Decision: {'CONFIRMED' if success_rate > 0 else 'REJECTED'}") 